# Notebook 2: Stock Price Data & Return Calculation

Fetching historical stock prices via `yfinance` for all 4 tickers,
then calculates post-earnings returns at 1-day, 3-day, and 5-day horizons.

**Output:** `data/processed/price_returns.csv`

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

In [1]:
RAW_DIR       = 'data/raw/prices'
PROCESSED_DIR = 'data/processed'
os.makedirs(RAW_DIR, exist_ok=True)

TICKERS = ['AAPL', 'MSFT', 'NVDA', 'JPM']

SECTOR_MAP = {
    'AAPL': 'Technology', 'MSFT': 'Technology', 'NVDA': 'Technology',
    'JPM':  'Finance'
}

EARNINGS_DATES = {
    'AAPL': [
        ('2022-01-28', 'Q1', '2022'), # year 2022
        ('2022-04-29', 'Q2', '2022'),
        ('2022-07-28', 'Q3', '2022'),
        ('2022-10-27', 'Q4', '2022'),
    ],
    'MSFT': [
        ('2023-10-24', 'Q1', '2024'),  # year 2024
        ('2024-01-31', 'Q2', '2024'),
        ('2024-04-25', 'Q3', '2024'),
        ('2024-07-30', 'Q4', '2024'),
    ],
    'NVDA': [
        ('2024-05-29', 'Q1', '2025'),  # year 2025
        ('2024-08-28', 'Q2', '2025'),
        ('2024-11-20', 'Q3', '2025'),
        ('2025-02-26', 'Q4', '2025'),
    ],
    'JPM': [
        ('2024-04-12', 'Q1', '2024'),  # year 2024
        ('2024-07-12', 'Q2', '2024'),
        ('2024-10-11', 'Q3', '2024'),
        ('2025-01-15', 'Q4', '2024'),
    ],
}

NameError: name 'os' is not defined

## 1. Download Full Year of Price Data

In [3]:
# ── 1. Download prices ───────────────────────────────────────────────────────

def download_prices(tickers: list, start: str = '2021-12-01', end: str = '2025-03-13') -> pd.DataFrame:
    """
    Download daily adjusted close prices for a list of tickers.

    Args:
        tickers: List of ticker strings
        start:   Start date string 'YYYY-MM-DD'
        end:     End date string 'YYYY-MM-DD'

    Returns:
        DataFrame with Date index and one column per ticker (adjusted close)
    """
    print(f'Downloading price data for: {tickers}')
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)

    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw['Close']
    else:
        prices = raw[['Close']]
        prices.columns = tickers

    prices.index = pd.to_datetime(prices.index)
    print(f'Downloaded {len(prices)} trading days, {prices.shape[1]} tickers')
    print(f'Date range: {prices.index.min().date()} → {prices.index.max().date()}')
    return prices


prices = download_prices(TICKERS)
prices.to_csv(os.path.join(RAW_DIR, 'daily_prices.csv'))
print('Raw prices saved.')
print(prices.tail(5))

Downloaded 822 trading days, 4 tickers
Date range: 2021-12-01 → 2025-03-12
Raw prices saved.
Ticker            AAPL         JPM        MSFT        NVDA
Date                                                      
2025-03-06  234.308762  241.586349  393.874664  110.533783
2025-03-07  238.032547  237.411942  390.321869  112.653091
2025-03-10  226.492844  227.554077  377.271759  106.944962
2025-03-11  219.881653  224.535965  377.559570  108.724373
2025-03-12  216.038422  223.320877  380.358124  115.712738


## 2. Calculate Post-Earnings Returns

For each earnings date we compute:
- **1-day return:** close(t+1) / close(t) - 1  
- **3-day return:** close(t+3) / close(t) - 1  
- **5-day return:** close(t+5) / close(t) - 1  

Where `t` = the earnings announcement date (close price that day).

In [4]:
# ── 2. Calculate post-earnings returns ──────────────────────────────────────

def get_nth_trading_day_return(prices: pd.DataFrame, ticker: str,
                               earnings_date: str, n: int) -> float | None:
    """
    Calculate n-day post-earnings return for a ticker.

    Finds the closest trading day on or after earnings_date as t=0,
    then computes return over n trading days.

    Returns:
        Float return (e.g. 0.032 = 3.2%) or None if data unavailable.
    """
    ticker_prices = prices[ticker].dropna()
    target        = pd.Timestamp(earnings_date)

    future_dates = ticker_prices.index[ticker_prices.index >= target]
    if len(future_dates) < n + 1:
        return None

    t0_date = future_dates[0]
    tn_date = future_dates[n]

    p0 = ticker_prices.loc[t0_date]
    pn = ticker_prices.loc[tn_date]

    return round((pn / p0) - 1, 6)

In [6]:
records = []
for ticker, dates in EARNINGS_DATES.items():
    for date_str, quarter, fiscal_year in dates:
        r1 = get_nth_trading_day_return(prices, ticker, date_str, 1)
        r3 = get_nth_trading_day_return(prices, ticker, date_str, 3)
        r5 = get_nth_trading_day_return(prices, ticker, date_str, 5)

        records.append({
            'ticker':      ticker,
            'date':        date_str,
            'quarter':     quarter,
            'fiscal_year': fiscal_year,
            'sector':      SECTOR_MAP[ticker],
            'return_1d':   r1,
            'return_3d':   r3,
            'return_5d':   r5,
        })

df_returns = pd.DataFrame(records)
df_returns['date'] = pd.to_datetime(df_returns['date'])
print(df_returns.to_string(index=False))

ticker       date quarter fiscal_year     sector  return_1d  return_3d  return_5d
  AAPL 2022-01-28      Q1        2022 Technology   0.026126   0.032349   0.013384
  AAPL 2022-04-29      Q2        2022 Technology   0.001966   0.053092  -0.000882
  AAPL 2022-07-28      Q3        2022 Technology   0.032793   0.016905   0.053766
  AAPL 2022-10-27      Q4        2022 Technology   0.075552   0.040400  -0.040884
  MSFT 2023-10-24      Q1        2024 Technology   0.030678  -0.002178   0.022933
  MSFT 2024-01-31      Q2        2024 Technology   0.015594   0.020298   0.041426
  MSFT 2024-04-25      Q3        2024 Technology   0.018244  -0.024333  -0.003007
  MSFT 2024-07-30      Q4        2024 Technology  -0.010806  -0.034120  -0.055117
  NVDA 2024-05-29      Q1        2025 Technology  -0.037666   0.001524   0.066318
  NVDA 2024-08-28      Q2        2025 Technology  -0.063848  -0.140196  -0.146485
  NVDA 2024-11-20      Q3        2025 Technology   0.005347  -0.067654  -0.072315
  NVDA 2025-02-2

## 3. Flag Beat vs. Miss

A simple proxy: if 1-day return > 0 → stock "beat" expectations (market rewarded it).
This is a common approach in academic event studies when EPS estimates aren't available.

In [7]:
df_returns['beat_miss'] = df_returns['return_1d'].apply(
    lambda r: 'Beat' if r is not None and r > 0 else 'Miss'
)

print('\nBeat vs Miss distribution:')
print(df_returns['beat_miss'].value_counts())

print('\nAverage 1-day return by outcome:')
print(df_returns.groupby('beat_miss')['return_1d'].mean().round(4))



Beat vs Miss distribution:
beat_miss
Beat    11
Miss     5
Name: count, dtype: int64

Average 1-day return by outcome:
beat_miss
Beat    0.0218
Miss   -0.0401
Name: return_1d, dtype: float64


## 4. Summary Statistics

In [8]:
print('\n── Return Summary by Ticker ─────────────────────────────')
print(df_returns.groupby('ticker')[['return_1d', 'return_3d', 'return_5d']]
      .mean().round(4).to_string())

print('\n── Return Summary by Sector ─────────────────────────────')
print(df_returns.groupby('sector')[['return_1d', 'return_3d', 'return_5d']]
      .mean().round(4).to_string())

print('\n── Return Summary by Quarter ────────────────────────────')
print(df_returns.groupby('quarter')[['return_1d', 'return_3d', 'return_5d']]
      .mean().round(4).to_string())


── Return Summary by Ticker ─────────────────────────────
        return_1d  return_3d  return_5d
ticker                                 
AAPL       0.0341     0.0357     0.0063
JPM        0.0074     0.0229     0.0270
MSFT       0.0134    -0.0101     0.0016
NVDA      -0.0452    -0.0844    -0.0647

── Return Summary by Sector ─────────────────────────────
            return_1d  return_3d  return_5d
sector                                     
Finance        0.0074     0.0229     0.0270
Technology     0.0008    -0.0196    -0.0189

── Return Summary by Quarter ────────────────────────────
         return_1d  return_3d  return_5d
quarter                                 
Q1          0.0049     0.0042     0.0298
Q2         -0.0053    -0.0021    -0.0206
Q3          0.0132    -0.0173    -0.0019
Q4         -0.0031    -0.0206    -0.0371


## 5. Save

In [9]:
out_path = os.path.join(PROCESSED_DIR, 'price_returns.csv')
df_returns.to_csv(out_path, index=False)
print(f'\nSaved {len(df_returns)} rows → {out_path}')


Saved 16 rows → data/processed/price_returns.csv
